# Instrument pricing and value at risk

This notebook carries forward the calculations from the original `varprices.ipynb` using the three-column pricing snapshot already available in `app-var`. It runs with Python's standard library. Saved outputs and the original raw export are excluded.


In [ ]:
import csv
from collections import defaultdict
from math import exp, floor, pi, sqrt
from pathlib import Path
from statistics import NormalDist, mean, stdev

root = Path.cwd()
if not (root / "public").is_dir():
    root = root.parent
with (root / "public/prices/instrument_data.csv").open(newline="") as file:
    prices = defaultdict(dict)
    for row in csv.DictReader(file):
        prices[row["instrumentsymbol"].strip()][row["tradedate"].strip()] = float(row["midprice"])

def returns_by_interval(history):
    ordered = sorted(history.items())
    return {
        (start, end): current / previous - 1
        for (start, previous), (end, current) in zip(ordered, ordered[1:])
    }

returns = {symbol: returns_by_interval(history) for symbol, history in prices.items()}
print(f"{len(prices)} instruments")


## Per-instrument workings

Calculate simple mid-price returns, the 5th and 95th return percentiles, the 95th percentile of absolute returns, and a normal-approximation confidence interval for the mean. The confidence interval describes uncertainty about the mean return. VaR measures a loss quantile and is calculated separately.


In [ ]:
def percentile(values, probability):
    ordered = sorted(values)
    rank = probability * (len(ordered) - 1)
    lower = floor(rank)
    share = rank - lower
    return (1 - share) * ordered[lower] + share * ordered[min(lower + 1, len(ordered) - 1)]

def instrument_statistics(history):
    pct_returns = [value * 100 for value in history.values()]
    n = len(pct_returns)
    average, deviation = mean(pct_returns), stdev(pct_returns)
    margin = 1.96 * deviation / sqrt(n)
    return {
        "observations": n,
        "mean_pct": average,
        "sample_sd_pct": deviation,
        "p05_pct": percentile(pct_returns, 0.05),
        "p95_pct": percentile(pct_returns, 0.95),
        "absolute_p95_pct": percentile([abs(value) for value in pct_returns], 0.95),
        "mean_ci95_pct": (average - margin, average + margin),
    }

statistics = {
    symbol: instrument_statistics(history)
    for symbol, history in returns.items() if len(history) >= 2
}
statistics["EURUSD"]


## Portfolio workings

Use exactly matching observation intervals, apply constant hypothetical USD exposures, and convert P/L into losses. Historical VaR is the linearly interpolated 95th loss percentile. Historical expected shortfall averages the worst 5% of probability mass with partial weight on the boundary observation. The parametric model assumes a zero mean and uses the sample standard deviation of P/L.


In [ ]:
positions = {"EURUSD": 60_000, "USDJPY": 40_000}
intervals = sorted(set.intersection(*(set(returns[symbol]) for symbol in positions)))
pnl = [sum(position * returns[symbol][interval] for symbol, position in positions.items())
       for interval in intervals]
losses = sorted((-value for value in pnl), reverse=True)
tail_probability = 0.05
tail_count = len(losses) * tail_probability
whole = floor(tail_count)
historical_es = (sum(losses[:whole]) + losses[whole] * (tail_count - whole)) / tail_count

sigma = stdev(pnl)
z = NormalDist().inv_cdf(0.95)
normal_tail_mean = exp(-z**2 / 2) / (sqrt(2 * pi) * tail_probability)
results = {
    "matched_intervals": len(intervals),
    "first_date": intervals[0][0],
    "last_date": intervals[-1][1],
    "sample_pnl_sd_usd": sigma,
    "parametric_var_usd": z * sigma,
    "parametric_es_usd": normal_tail_mean * sigma,
    "historical_var_usd": percentile(losses, 0.95),
    "historical_es_usd": historical_es,
}
print(results)
